In [5]:
!pip install -q datasets huggingface_hub scikit-learn tqdm pandas requests


In [6]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfApi, HfFolder

# Correct FPB dataset id
DATASET_NAME = "ChanceFocus/en-fpb"

# Get HF token: env -> Colab userdata -> manual
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf_...): ")

# Verify access (optional but helpful)
api = HfApi()
info = api.dataset_info(DATASET_NAME, token=hf_token)
HfFolder.save_token(hf_token)
print(f"Access to {DATASET_NAME} OK ✅")
print("Files/siblings:", [s.rfilename for s in info.siblings])

# Load all splits
ds_all = load_dataset(DATASET_NAME, token=hf_token)

print("\nLoaded splits:")
for split_name, split_ds in ds_all.items():
    print(f"  {split_name}: {len(split_ds)} examples")

# Prefer 'test' if available, else first split
SPLIT = "test" if "test" in ds_all else list(ds_all.keys())[0]
dataset = ds_all[SPLIT]

print(f"\nUsing split: {SPLIT}")
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


Access to ChanceFocus/en-fpb OK ✅
Files/siblings: ['.gitattributes', 'README.md', 'data/test-00000-of-00001-8bd1e21c671fb670.parquet', 'data/train-00000-of-00001-ab9a3b4799b09589.parquet', 'data/valid-00000-of-00001-303e4ba2afe838d4.parquet']

Loaded splits:
  train: 3100 examples
  test: 970 examples
  valid: 776 examples

Using split: test
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', '

In [7]:
import re
from typing import Optional

from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: Optional[str]) -> str:
    """
    Map a free-form model output to one of the three labels.
    """
    if raw is None:
        return "neutral"
    text = raw.strip().lower()

    # Exact match
    for label in LABELS:
        if text == label:
            return label

    # Whole-word match
    for label in LABELS:
        if re.search(r"\b" + re.escape(label) + r"\b", text):
            return label

    # Fallback
    return "neutral"


In [8]:
import time
import requests

# 1) Set your Grok API key (do NOT put any other provider's key here)
if "GROK_API_KEY" not in os.environ:
    os.environ["GROK_API_KEY"] = getpass.getpass("Grok API key (sk-...): ")

GROK_API_KEY = os.environ["GROK_API_KEY"]

# 2) Configure base URL and model id according to your Grok provider
API_BASE = "https://api.x.ai"  # change if your provider uses a different base URL
GROK_MODEL_ID = "grok-4"       # change to the exact model id you want to evaluate

SYSTEM_PROMPT_GROK = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog,\n"
    "classify its overall sentiment from the perspective of an investor as\n"
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_grok4(
    sentence: str,
    max_retries: int = 6,
    sleep_base: float = 1.0,
) -> str:
    """
    Call Grok-4 once via HTTP and return a normalized three-way label.
    """
    last_err = None

    for attempt in range(max_retries):
        try:
            url = f"{API_BASE}/v1/chat/completions"
            headers = {
                "Authorization": f"Bearer {GROK_API_KEY}",
                "Content-Type": "application/json",
            }
            payload = {
                "model": GROK_MODEL_ID,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_GROK},
                    {"role": "user", "content": sentence},
                ],
                "max_tokens": 16,
                "temperature": 0.0,
            }

            resp = requests.post(url, headers=headers, json=payload, timeout=60)
            if resp.status_code != 200:
                raise RuntimeError(
                    f"HTTP {resp.status_code}: {resp.text[:300]}"
                )

            data = resp.json()

            # Basic chat-completions-compatible parsing
            raw_content = data["choices"][0]["message"]["content"]
            # Some providers may return a list of parts instead of a plain string
            if isinstance(raw_content, list):
                text_out = "".join(
                    part.get("text", "") if isinstance(part, dict) else str(part)
                    for part in raw_content
                )
            else:
                text_out = str(raw_content)

            return normalize_prediction(text_out)

        except Exception as e:
            last_err = e
            wait = sleep_base * (2 ** attempt)
            print(f"[Grok-4] Error on attempt {attempt+1}/{max_retries}: {last_err}")
            time.sleep(wait)

    print("[Grok-4] Failed after retries, returning 'neutral'.")
    return "neutral"


Grok API key (sk-...): ··········


In [9]:
y_true = []
y_pred = []

print(f"Evaluating model: {GROK_MODEL_ID} on {len(dataset)} FPB examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]               # change this if your column name is different
    true_label = example["answer"].lower().strip()  # change if your label column has another name
    pred_label = classify_with_grok4(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: grok-4 on 970 FPB examples...



 34%|███▍      | 333/970 [44:51<1:26:33,  8.15s/it]

[Grok-4] Error on attempt 1/6: HTTPSConnectionPool(host='api.x.ai', port=443): Read timed out. (read timeout=60)


 76%|███████▌  | 734/970 [1:57:50<49:28, 12.58s/it]

[Grok-4] Error on attempt 1/6: HTTPSConnectionPool(host='api.x.ai', port=443): Read timed out. (read timeout=60)


100%|██████████| 970/970 [2:42:58<00:00, 10.08s/it]

Total examples: 970
Got predictions for: 970


In [10]:
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.7361

Classification report:
              precision    recall  f1-score   support

    negative       0.76      0.98      0.86       116
     neutral       0.96      0.58      0.73       577
    positive       0.56      0.95      0.70       277

    accuracy                           0.74       970
   macro avg       0.76      0.84      0.76       970
weighted avg       0.82      0.74      0.74       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functio